In [6]:
import re
import string
import nltk
from nltk.corpus import stopwords
nltk.download('punkt')
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
from nltk import WordNetLemmatizer
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()
nltk.download('wordnet')
nltk.download('omw-1.4')
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer 

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [7]:
data= pd.read_csv('sentences.csv')
data.head()

,id,sentence
0,1,the cat sat on the mat near the window
1,2,a dog barked loudly in the quiet neighborhood
2,3,she opened the door and walked into the room
3,4,the sun rises in the east every morning
4,5,he read the book from cover to cover


In [8]:
def preprocessing_pipeline(text):
    text=text.lower()
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F" 
        "\U0001F300-\U0001F5FF" 
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "\U0001f926-\U0001f937"
        "\U00010000-\U0010ffff"
        "\u2640-\u2642"
        "\u2600-\u2B55"
        "\u200d"
        "\u23cf"
        "\u23e9"
        "\u231a"
        "\ufe0f" 
        "\u3030"
        "]+", re.UNICODE)
    text = emoji_pattern.sub(r'', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    filtered_tokens = [word for word in tokens if word not in stop_words]
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in filtered_tokens]
    preprocessed_text = ' '.join(lemmatized_tokens)
    return preprocessed_text
data['preprocessed_text'] = data['sentence'].apply(preprocessing_pipeline)
data.head()

,id,sentence,preprocessed_text
0,1,the cat sat on the mat near the window,cat sat mat near window
1,2,a dog barked loudly in the quiet neighborhood,dog barked loudly quiet neighborhood
2,3,she opened the door and walked into the room,opened door walked room
3,4,the sun rises in the east every morning,sun rise east every morning
4,5,he read the book from cover to cover,read book cover cover


In [9]:
tfidf = TfidfVectorizer(max_features=5000)
text_matrix = tfidf.fit_transform(data['preprocessed_text'])
print(text_matrix)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 857 stored elements and shape (170, 544)>
  Coords	Values
  (0, 66)	0.42787128230247584
  (0, 400)	0.42787128230247584
  (0, 282)	0.49023821130106804
  (0, 300)	0.49023821130106804
  (0, 528)	0.39138896755316055
  (1, 120)	0.44098647406494756
  (1, 26)	0.47644211765941763
  (1, 274)	0.44098647406494756
  (1, 369)	0.3963176458649678
  (1, 304)	0.47644211765941763
  (2, 320)	0.5167359392517009
  (2, 121)	0.49248833873137426
  (2, 511)	0.5167359392517009
  (2, 393)	0.47267660655561977
  (3, 466)	0.48862456913432484
  (3, 386)	0.48862456913432484
  (3, 130)	0.48862456913432484
  (3, 139)	0.40645134376959086
  (3, 292)	0.3442896984557921
  (4, 377)	0.378483973033616
  (4, 38)	0.32331578566130814
  (4, 99)	0.8673043208118476
  (5, 75)	0.41067061258154497
  (5, 353)	0.44895021622024256
  (5, 322)	0.47611002723035667
  :	:
  (165, 302)	0.46060917286746544
  (165, 491)	0.46060917286746544
  (166, 89)	0.3530880348372397
  (166, 387)	0

In [11]:
# Check available columns in the dataset
print("Available columns:", data.columns.tolist())
print("\nDataFrame shape:", data.shape)
print("\nFirst few rows:")
print(data.head())

Available columns: ['id', 'sentence', 'preprocessed_text']

DataFrame shape: (170, 3)

First few rows:
   id                                       sentence  \
0   1         the cat sat on the mat near the window   
1   2  a dog barked loudly in the quiet neighborhood   
2   3   she opened the door and walked into the room   
3   4        the sun rises in the east every morning   
4   5           he read the book from cover to cover   

                      preprocessed_text  
0               cat sat mat near window  
1  dog barked loudly quiet neighborhood  
2               opened door walked room  
3           sun rise east every morning  
4                 read book cover cover  


In [12]:
# Create synthetic labels for demonstration (0: negative, 1: positive)
# In a real scenario, you'd have actual labels or use a labeled dataset
import numpy as np
np.random.seed(42)  # For reproducible results
data['label'] = np.random.randint(0, 2, size=len(data))

print("Labels distribution:")
print(data['label'].value_counts())
print("\nDataset with labels:")
print(data.head())

Labels distribution:
label
1    90
0    80
Name: count, dtype: int64

Dataset with labels:
   id                                       sentence  \
0   1         the cat sat on the mat near the window   
1   2  a dog barked loudly in the quiet neighborhood   
2   3   she opened the door and walked into the room   
3   4        the sun rises in the east every morning   
4   5           he read the book from cover to cover   

                      preprocessed_text  label  
0               cat sat mat near window      0  
1  dog barked loudly quiet neighborhood      1  
2               opened door walked room      0  
3           sun rise east every morning      0  
4                 read book cover cover      0  


In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(text_matrix, data['label'], test_size=0.2, random_state=42)

KeyError: 'label'

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


In [ ]:
from sklearn.metrics import confusion_matrix, recall_score, precision_score, f1_score
cm = confusion_matrix(y_test, y_pred)
recall = recall_score(y_test, y_pred, average='weighted')
precision = precision_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
print("Confusion Matrix:\n", cm)
print("Recall:", recall)
print("Precision:", precision)
print("F1 Score:", f1)
